In [1]:
import socket
import cv2
import pickle
import struct
import numpy as np

# Receiver configuration
host_ip = '0.0.0.0'
port = 8485
is_receiving = True

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.bind((host_ip, port))
server_socket.listen(5)
print(f"🟢 Listening on {host_ip}:{port}...")

conn, addr = server_socket.accept()
print(f"✅ Connected by {addr}")

🟢 Listening on 0.0.0.0:8485...
✅ Connected by ('10.0.0.49', 53052)


In [2]:


data = b""
payload_size = struct.calcsize("Q")

try:
    while True:
        # Toggle logic (simulate external control; you can replace with actual control later)
        cmd = input("Type 'r' to RECEIVE or 'p' to PAUSE streaming: ").strip().lower()
        if cmd == 'r':
            is_receiving = True
            conn.sendall(b'READY')
        elif cmd == 'p':
            is_receiving = False
            conn.sendall(b'PAUSE')
        else:
            print("⛔ Unknown command")

        if not is_receiving:
            continue  # Skip frame reception

        while len(data) < payload_size:
            packet = conn.recv(4096)
            if not packet:
                break
            data += packet

        if len(data) < payload_size:
            continue

        packed_msg_size = data[:payload_size]
        data = data[payload_size:]
        msg_size = struct.unpack("Q", packed_msg_size)[0]

        while len(data) < msg_size:
            data += conn.recv(4096)

        frame_data = data[:msg_size]
        data = data[msg_size:]

        frame = cv2.imdecode(np.frombuffer(frame_data, dtype=np.uint8), cv2.IMREAD_COLOR)
        if frame is not None:
            cv2.imshow("📷 Live Stream", frame)

        if cv2.waitKey(1) == ord('q'):
            break

except Exception as e:
    print("❌ Error:", e)
finally:
    conn.close(
    server_socket.close()
    cv2.destroyAllWindows()
    print("🔌 Connection closed.")


⛔ Unknown command
🔌 Connection closed.


KeyboardInterrupt: Interrupted by user